# Muhr — E9: regeneratsiya hujumlari (IEEE TIFS uchun)

Bitta bloknot, bitta ishga tushirish. Regeneratsiya (tozalash) hujumlari ostida tekshiriladi: bizning tarmoq + G9, Pixel Seal + G9, Video Seal + G9 hamda TrustMark, MBRS va WAM (o'z holida).

**Hujumlar** (`muhr_exp/regen.py`):
- `Regen-ctrl`: faqat 512×512 ga o'lchamni o'zgartirib, qaytarish (nazorat).
- `Regen-VAE`: Stable Diffusion KL-VAE; o'rganilgan kodeklar bmshj2018 va cheng2020 (quality 3).
- `Regen-Diff`: SD img2img, strength 0.1 va 0.2 (Zhao et al., 2024).

**Input (Add Input):** `muhr-v2` (**2.6** versiya), `best.pt` manbasi, COCO 2017 (`awsaf49/coco-2017-dataset`).
**Sozlamalar:** GPU T4, **Internet: On** (Hugging Face, GitHub va compressai og'irliklari uchun).
**Hajm:** T-COCO'ning birinchi 200 tasviri (bl_marked bilan bir xil tasvirlar). Taxminiy vaqt ≈ 1–1.5 GPU-soat [taxmin].

Log boshida `regen preflight: {...}` qatori chiqadi. Agar unda `DROPPED` bo'lsa, o'sha hujum o'tkazib yuboriladi, qolganlari ishlayveradi. Shu qatorni menga yuboring.
Agar sessiya vaqti tugab qolsa: shu versiya chiqishini Add Input qilib, qayta Commit qiling. Tayyor bo'laklar qayta hisoblanmaydi.


In [ ]:
# ===== Sozlamalar =====
STAGE = "e9_regen"
N_IMAGES = 200
MAX_HOURS = 11.0
ATTACKS = "Regen-ctrl,Regen-VAE,Regen-Diff"
VS_COMMIT = "f12875a9b5e711e0c6e2dc3a22aca18231ce4a1a"           # Video Seal kodi (bl_* bilan bir xil)
HF_REPO, HF_REV = "tangtianzhong/img-wm-torchscript", "845dc751783db2a03a4b14ea600b0a4a9aba89aa"


In [ ]:
import glob, os, shutil, subprocess, sys
from pathlib import Path
W, IN, TMP = Path("/kaggle/working"), Path("/kaggle/input"), Path("/kaggle/tmp")
TMP.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise SystemExit(f"buyruq xato bilan tugadi: {cmd[:3]}")

def find_dir(name):
    hits = [h for h in sorted(glob.glob(str(IN / "**" / name), recursive=True)) if os.path.isdir(h)]
    return Path(hits[0]) if hits else None

def _ver(d):
    try:
        return tuple(int(x) for x in (d / "VERSION").read_text().strip().split("."))
    except Exception:
        return (0,)
src = sorted({Path(p).parents[1] for p in glob.glob(str(IN / "**/muhr_proj/muhr_exp/codec2.py"), recursive=True)},
             key=_ver, reverse=True)
assert src, "muhr-v2 dataseti (muhr_proj/muhr_exp) topilmadi — Add Input qiling"
print("kod:", src[0], "versiya", ".".join(map(str, _ver(src[0]))))
assert _ver(src[0]) >= (2, 6), "muhr-v2 eski versiyada — Input'da 2.6 versiyani tanlang"
CODE = W / "muhr_proj"
if CODE.exists():
    shutil.rmtree(CODE)
shutil.copytree(src[0], CODE)
ck = sorted(glob.glob(str(IN / "**/runs/taklif/best.pt"), recursive=True))
assert ck, "runs/taklif/best.pt topilmadi"
CKPT = ck[0]; print("checkpoint:", CKPT)
run([sys.executable, "-m", "pip", "install", "-q", "galois", "lpips", "bchlib", "omegaconf", "einops", "pytorch_msssim",
     "pycocotools", "av", "decord", "diffusers", "compressai", "transformers>=4.41.2", "accelerate"])


In [ ]:
# --- Video Seal kodi va baseline og'irliklari
VS = TMP / "videoseal"
if not VS.exists():
    run(["git", "clone", "-q", "https://github.com/facebookresearch/videoseal", VS])
    run(["git", "checkout", "-q", VS_COMMIT], cwd=VS)
(VS / "ckpts").mkdir(exist_ok=True)
if not (VS / "ckpts" / "hidden_encoder_48b.pt").exists():
    from huggingface_hub import snapshot_download
    try:
        d = snapshot_download(HF_REPO, revision=HF_REV, local_dir=str(TMP / "hf_ts"))
    except Exception as e:
        print("pinned revision failed:", e, "-> main"); d = snapshot_download(HF_REPO, local_dir=str(TMP / "hf_ts"))
    for f in glob.glob(str(Path(d) / "**" / "*.pt"), recursive=True):
        shutil.copy(f, VS / "ckpts" / Path(f).name)
print(sorted(os.listdir(VS / "ckpts")))


In [ ]:
# --- ma'lumotlar: T-COCO (val2017[1500:3500]) ning birinchi N_IMAGES tasviri
def link_subset(files, dst):
    dst = Path(dst)
    if dst.exists():
        return dst
    dst.mkdir(parents=True)
    for f in files:
        os.symlink(f, dst / Path(f).name)
    return dst
val = find_dir("val2017"); assert val is not None, "COCO val2017 topilmadi"
vfiles = sorted(glob.glob(str(val / "*.jpg")))
T_COCO = link_subset(vfiles[1500:3500], TMP / "test_coco")

OUT = W / STAGE
OUT.mkdir(exist_ok=True)
prev = glob.glob(str(IN / "**" / f"{STAGE}.zip"), recursive=True)
if prev and not any(OUT.iterdir()):
    print("oldingi natija topildi:", prev[0]); shutil.unpack_archive(prev[0], OUT)
print("mavjud bo'laklar:", len(glob.glob(str(OUT / "**" / "chunk_*.npz"), recursive=True)))


In [ ]:
# --- regeneratsiya modellarini oldindan tekshirish (SD va compressai og'irliklari shu yerda yuklanadi)
os.chdir(CODE)
pre = subprocess.run([sys.executable, "-c", "from muhr_exp import regen; import json; print(json.dumps(regen.register()['preflight']))"],
                     capture_output=True, text=True)
print(pre.stdout[-3000:], pre.stderr[-2000:])
assert '"Regen-Diff:0.1": "ok"' in pre.stdout and '"Regen-VAE:kl": "ok"' in pre.stdout,     "Stable Diffusion yuklanmadi — Internet: On ekanini tekshiring va log'ni menga yuboring"


In [ ]:
os.chdir(CODE)
py = sys.executable
reg = ["--regen", "--attacks", ATTACKS, "--max-hours", MAX_HOURS]
# 1) bizning tarmoq + G9 (global rejim)
run([py, "scripts/e3_marked.py", "--ckpt", CKPT, "--data", T_COCO, "--n", N_IMAGES, "--designs", "G",
     "--modes", "global", "--chunk", 50, "--out", OUT / "ours", *reg])
# 2) G9 kodek Pixel Seal va Video Seal ichida
run([py, "scripts/e2_baselines.py", "--stage", "marked", "--codec", "G9", "--methods", "pixelseal,videoseal",
     "--vs-root", VS, "--ckpt", CKPT, "--data", T_COCO, "--n", N_IMAGES, "--chunk", 50, "--out", OUT / "codec", *reg])
# 3) baseline'lar o'z holida (bit-chegara qoidasi)
run([py, "scripts/e2_baselines.py", "--stage", "marked", "--methods", "trustmark,mbrs,wam",
     "--vs-root", VS, "--ckpt", CKPT, "--data", T_COCO, "--n", N_IMAGES, "--chunk", 50, "--out", OUT / "native", *reg])


In [ ]:
os.chdir(W)
z = shutil.make_archive(str(W / STAGE), "zip", OUT)
print("saqlandi:", z, round(os.path.getsize(z) / 1e6, 1), "MB")
print("bo'laklar:", len(glob.glob(str(OUT / "**" / "chunk_*.npz"), recursive=True)))
